# Phase 0 — Model Pilot Test
Qwen2.5-7B-Instruct vs Llama-3.1-8B-Instruct tool-calling reliability check.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

In [ ]:
!nvidia-smi

## 1. Clone the repo

In [ ]:
!git clone https://github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune

If the repo is private, use this instead of the plain clone above:
```python
from getpass import getpass
token = getpass('Paste your GitHub token: ')
!git clone https://{token}@github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune
```

## 2. Install dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes

## 3. Hugging Face login (needed for gated Llama-3.1-8B-Instruct)

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import whoami
print(whoami())

## 4. Quick single-prompt sanity check (run this first — takes ~1-2 min)
Confirms loading + generation both work before running the full pilot script.

In [ ]:
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

t0 = time.time()
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct", quantization_config=bnb_config, device_map="auto"
)
print(f"Load time: {time.time()-t0:.1f}s")

t0 = time.time()
inputs = tok("What's the weather in Tokyo?", return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=50)
print(f"Generate time: {time.time()-t0:.1f}s")
print(tok.decode(out[0], skip_special_tokens=True))

import gc
del model
gc.collect()
torch.cuda.empty_cache()

If load time is under ~1 min and generate time is under ~30s with no 'offloaded to cpu and disk' warning, you're good to run the full pilot script below.

## 5. Run the full pilot script (both models, 10 prompts each)

In [ ]:
!git pull

In [ ]:
!python envs/pilot_test.py

## 6. Check results

In [ ]:
import json
with open("results/pilot_results.json") as f:
    results = json.load(f)
for model_name, outputs in results.items():
    print(f"\n=== {model_name} ===")
    for r in outputs:
        print(f"Prompt: {r['prompt']}")
        print(f"Output: {r['output'][:200]}")
        print("---")

## 7. Push results back to GitHub
Run this locally in VSCode terminal, not in Colab (Colab won't have your git push credentials set up unless you configured the token method above):
```bash
git add results/pilot_results.json
git commit -m "Phase 0: pilot results"
git push
```